# 03 · ephys membrane test: Cm, Ra, Rt, Rm and tau

Input: `02_ephys_anomaly-fix/output/analysis_ready.csv` (only recordings whose problems are resolved).
Output: **only the final values**, one row per recording, in `output/membrane_test_values.csv`.

**Method** (Clampex Membrane Test algorithm, pCLAMP 10 User Guide pp. 163-166), for every pulse edge:

| quantity | definition |
|---|---|
| Rt | ΔV / ΔI_steady-state (= Ra + Rm) |
| tau | single-exponential fit of the current transient from 80 % to 10 % of its peak |
| Qt | charge of the transient + ΔI·tau |
| Ra | tau·ΔV / Qt |
| Rm | Rt − Ra |
| Cm | Qt·Rt / (ΔV·Rm) |

The final value per recording is the **median over valid pulse edges**. ⚠️ Not yet validated against
Clampex readouts; the choice of a single- vs two-exponential model for Ra is still open (see README).

In [ ]:
OVERWRITE = True   # replace an existing membrane_test_values.csv

import pandas as pd
from IPython.display import Markdown, display
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 200)
from ephys_analysis.membrane_test import MTSettings, analyse_recording, analyse_table, plot_recording
from ephys_analysis.utils import RunLog, check_writable, find_repo_root, load_config, repo_path, stamp, write_csv

REPO = find_repo_root()
cfg = load_config(REPO / "03_ephys_membrane-test")
settings = MTSettings.from_config(cfg["membrane_test"])
out = cfg["_project_dir"] / "output"
target = out / "membrane_test_values.csv"
check_writable([target], OVERWRITE)
ready = pd.read_csv(repo_path(cfg, cfg["recordings_table"]), dtype={"cell_tag": "string"})
selection = ready[ready["family"] == cfg["family"]]
display(selection[["abf_filename", "cell_tag", "current_source", "corrections"]])

## Step 1 · One recording in detail
How the analysis sees a single recording: the command channel it chose (by the evoked response, never by
channel number), every pulse edge with its values and QC flags, and a figure of the fits.

In [ ]:
example = selection.iloc[0]
pulses, info = analyse_recording(example["original_path"], settings, example["recording_id"])
print(info["command_dac_selection"])
display(pulses[["sweep", "edge_type", "valid", "flags", "dV_mV", "Rt_MOhm", "tau_ms", "Ra_MOhm", "Cm_pF"]].round(2))
fig = plot_recording(example["original_path"], pulses, info)

## Step 2 · All recordings

In [ ]:
log = RunLog(cfg, out, save=False)
values = analyse_table(selection, settings, log)
display(values[["abf_filename", "n_valid_pulses", "Cm_pF", "Ra_MOhm", "Rt_MOhm", "Rm_MOhm", "tau_fast_ms", "tau_slow_ms", "ra_fast_status",
                "Ra_clampex_MOhm", "qc_status"]].round(2))

## Step 3 · Save the final values

In [ ]:
write_csv(stamp(values, log.run_id), target)
print("written:", target)